# 第21章 資料を読んで答える：RAG

LLMにカンニングペーパーを持たせて、知らないことに答えさせる

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/21-rag.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 知らないことを聞いてみる


In [ ]:
!pip install -q sentence-transformers
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/rag_docs.jsonl
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16 if device == "cuda" else torch.float32).to(device)

def ask(messages, n=150):
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    out = llm.generate(**inputs, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

question = "ボンゴ情報専門学校の学食で一番人気のメニューは何？値段も教えて。"
print(ask([{"role": "user", "content": question}]))

## 資料を読み込む


In [ ]:
docs = [json.loads(l) for l in open("rag_docs.jsonl", encoding="utf-8")]
print(len(docs), "件")
for d in docs[:3]:
    print(d["id"], d["title"], "：", d["text"][:40], "…")

## 文章を1本のベクトルにする：文の埋め込み


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("intfloat/multilingual-e5-small", device=device)
doc_texts = [f"passage: {d['title']}。{d['text']}" for d in docs]
doc_vecs = embedder.encode(doc_texts, convert_to_tensor=True, normalize_embeddings=True)
print(doc_vecs.shape)

## 近い文章を探す：コサイン類似度


In [ ]:
import torch.nn.functional as F
a = F.normalize(torch.tensor([3.0, 4.0]), dim=0)    # 長さ1に
b = F.normalize(torch.tensor([6.0, 8.0]), dim=0)
c = F.normalize(torch.tensor([-4.0, 3.0]), dim=0)
print("同じ向き：", (a @ b).item(), " 直角：", round((a @ c).item(), 6))

In [ ]:
def search(query, k=3):
    q = embedder.encode([f"query: {query}"], convert_to_tensor=True, normalize_embeddings=True)
    scores = (q @ doc_vecs.T)[0]                     # 全資料とのコサイン類似度
    top = torch.topk(scores, k)
    return [(docs[i], s.item()) for s, i in zip(top.values, top.indices)]

for d, score in search(question):
    print(f"{score:.3f}  [{d['id']}] {d['title']}")

## 資料を渡して答えさせる


In [ ]:
def rag_answer(question, k=3):
    hits = search(question, k)
    context = "\n".join(f"[{d['id']}] {d['title']}：{d['text']}" for d, _ in hits)
    messages = [
        {"role": "system", "content": "あなたはボンゴ情報専門学校の案内係です。必ず与えられた資料だけを根拠に答え、"
                                      "根拠にした資料の番号を [番号] の形で示してください。資料に書いていないことは「資料にはありません」と答えてください。"},
        {"role": "user", "content": f"資料：\n{context}\n\n質問：{question}"},
    ]
    return ask(messages)

print(rag_answer(question))

In [ ]:
print(rag_answer("ボンゴ情報専門学校にプールはありますか？"))

## 長い資料は切り分ける：チャンク分割


In [ ]:
def chunk(text, size=100, overlap=20):
    """size文字ずつに切る。前のチャンクと overlap 文字だけ重ねる"""
    pieces = []
    start = 0
    while start < len(text):
        pieces.append(text[start:start + size])
        if start + size >= len(text):
            break
        start += size - overlap
    return pieces

long_text = "".join(d["text"] for d in docs[:3])
for i, c in enumerate(chunk(long_text)):
    print(i, len(c), c[:25], "…")

## 検索を評価する：Hit@k


In [ ]:
eval_set = [
    ("お昼休みは何時から？", 2), ("本は何冊まで借りられる？", 4), ("GPUの部屋の予約はいつまで？", 5),
    ("ボンゴ部の活動日は？", 6), ("学園祭はいつ？", 8), ("追試験を受けるには？", 9),
    ("基本情報に受かると何かもらえる？", 10), ("学生証をなくしたら？", 12), ("Wi-Fiの名前は？", 13),
    ("台風の日の授業は？", 14), ("卒業研究の発表は何分？", 16), ("消しゴムはいくら？", 18),
]

def hit_at_k(search_fn, k):
    hits = 0
    for q, answer_id in eval_set:
        found = [d["id"] for d, _ in search_fn(q, k)]
        hits += answer_id in found
    return hits / len(eval_set)

for k in [1, 3]:
    print(f"埋め込み検索 Hit@{k}：{hit_at_k(search, k):.2f}")

In [ ]:
def keyword_search(query, k=3):
    def bigrams(s):
        return {s[i:i + 2] for i in range(len(s) - 1)}
    q = bigrams(query)
    scored = [(d, len(q & bigrams(d["title"] + d["text"]))) for d in docs]
    scored.sort(key=lambda x: -x[1])
    return scored[:k]

for k in [1, 3]:
    print(f"キーワード検索 Hit@{k}：{hit_at_k(keyword_search, k):.2f}")